# ClassicNLG-Bench — Bộ Benchmark Truyền Thống cho Full Story (Đối chiếu với FullStory-Bench)

Đo lường độc lập trọn bộ **10 chỉ số NLG truyền thống** trên các file truyện dài đã sinh:
- **Trùng khớp Reference**: `BLEU-1`, `BLEU-2`, `BLEU-3`, `BLEU-4`, `ROUGE-L`
- **Đa dạng bề mặt**: `Distinct-1`, `Distinct-2`, `Distinct-3`, `Entropy-2` (Bigram Entropy)
- **Độ trôi chảy ngôn ngữ**: `Perplexity` (strided sliding window trên `gpt2` hoặc `Qwen/Qwen2.5-0.5B`)

Hỗ trợ đối chiếu tự động **Spearman $\rho$** với kết quả `FullStory-Bench` (`ConStory-Checker`, `EQ-Bench`, `DOC-NLI`) để kiểm chứng sự khác biệt giữa benchmark truyền thống và benchmark cốt truyện dài.

In [ ]:
# 1. Mount Google Drive & Sync Repository
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

REPO_DIR = Path('/content/MA-GRPO')
REPO_URL = 'github.com/AIVIETNAM-AIO-TruongDogki/MA_GRPO.git'
try:
    from google.colab import userdata
    GH_TOKEN = userdata.get('GH_TOKEN') or os.environ.get('GH_TOKEN', '')
except Exception:
    GH_TOKEN = os.environ.get('GH_TOKEN', '')

clone_url = f'https://{GH_TOKEN}@{REPO_URL}' if GH_TOKEN else f'https://{REPO_URL}'
if not REPO_DIR.exists():
    !git clone {clone_url} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only
%cd {REPO_DIR}

In [ ]:
# 2. Install Lightweight Dependencies
!pip install -q transformers accelerate matplotlib pandas

In [ ]:
# 3. Configure Story Inputs, Reference File & Optional FullStory-Bench Comparison
STORIES_FILES = [
    '/content/drive/MyDrive/MA-GRPO-ContextGate-Ablation/stories_7b_outputs.jsonl',
]
# Nếu chạy trên TELL ME A STORY test set (có gold reference 'targets'), trỏ vào đây để tính BLEU-1..4 & ROUGE-L thật:
REFERENCE_FILE = 'data/tell-me-a-story-test.jsonl'
# Nếu truyện sinh từ MoPS (không có gold reference): để 'none' (BLEU/ROUGE=N/A) hoặc 'premise_and_plan'
REFERENCE_FALLBACK = 'premise_and_plan'

PPL_MODEL = 'gpt2'  # Hoặc 'Qwen/Qwen2.5-0.5B'
FULLSTORY_RAW_JSONL = '/content/drive/MyDrive/MA-GRPO-Benchmarks/suite2_fullstory_bench/fullstory_raw.jsonl'
OUTPUT_DIR = '/content/drive/MyDrive/MA-GRPO-Benchmarks/classic_nlg_bench'
AUTO_DISCONNECT = False

In [ ]:
# 4. Run ClassicNLG-Bench via Subprocess
import subprocess
import sys

cmd = [
    sys.executable,
    'scripts/benchmark_classic_nlg.py',
    '--stories', *STORIES_FILES,
    '--output-dir', OUTPUT_DIR,
    '--reference-fallback', REFERENCE_FALLBACK,
    '--ppl-model', PPL_MODEL,
]
if REFERENCE_FILE and Path(REFERENCE_FILE).exists():
    cmd.extend(['--reference-file', REFERENCE_FILE])
if FULLSTORY_RAW_JSONL and Path(FULLSTORY_RAW_JSONL).exists():
    cmd.extend(['--compare-with-fullstory', FULLSTORY_RAW_JSONL])

print('Running:', ' '.join(cmd), flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f'benchmark_classic_nlg.py exited with code {ret}')

In [ ]:
# 5. Display Report & 4-Panel Visualization
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

out_path = Path(OUTPUT_DIR)
df = pd.read_csv(out_path / 'classic_nlg_summary.csv')
display(df)
display(Markdown((out_path / 'classic_nlg_report.md').read_text(encoding='utf-8')))

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
panels = [
    ('ROUGE-L', 'ROUGE-L (F1 %)'),
    ('Distinct-3 (%)', 'Distinct-3 (%)'),
    ('Entropy-2', 'Bigram Entropy-2 (bits)'),
    ('Perplexity', 'Perplexity (Lower=More Predictable)'),
]
for ax, (col, title) in zip(axes, panels):
    vals = df[col].fillna(0)
    ax.bar(df['arm'], vals, color='#805ad5')
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(out_path / 'classic_nlg_chart.png', dpi=150)
plt.show()

if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()